In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [2]:
df = pd.read_csv("diabetic_data.csv")

# Target
y = df["readmitted"].replace({
    "NO": 0,
    ">30": 1,
    "<30": 1
})

# Numeric features only
X = df.select_dtypes(include=["int64", "float64"])

# Train / Validation / Test Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

/tmp/ipykernel_10587/277306971.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df["readmitted"].replace({


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

baseline_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, y_pred)
    baseline_results[name] = auc_score

for model, score in baseline_results.items():
    print(model, "Validation AUC:", round(score, 3))

In [ ]:
#tuned logistic regression

lr_params = {
    "C": [0.01, 0.1, 1, 10]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    lr_params,
    scoring="roc_auc",
    cv=3
)

lr_grid.fit(X_train, y_train)

best_lr = lr_grid.best_estimator_
lr_pred = best_lr.predict_proba(X_val)[:, 1]
print("Tuned Logistic Regression AUC:", round(roc_auc_score(y_val, lr_pred), 3))

In [ ]:
# TUnes random forest

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    scoring="roc_auc",
    cv=3
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
rf_pred = best_rf.predict_proba(X_val)[:, 1]
print("Tuned Random Forest AUC:", round(roc_auc_score(y_val, rf_pred), 3))

In [ ]:
#tuned gradient boosting
gb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params,
    scoring="roc_auc",
    cv=3
)

gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
gb_pred = best_gb.predict_proba(X_val)[:, 1]
print("Tuned Gradient Boosting AUC:", round(roc_auc_score(y_val, gb_pred), 3))

In [ ]:
#tuned SMV
svm_params = {
    "C": [0.1, 1, 10]
}

svm_grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    svm_params,
    scoring="roc_auc",
    cv=3
)

svm_grid.fit(X_train, y_train)

best_svm = svm_grid.best_estimator_
svm_pred = best_svm.predict_proba(X_val)[:, 1]
print("Tuned SVM AUC:", round(roc_auc_score(y_val, svm_pred), 3))

In [ ]:
# Model comparison
results = {
    "Baseline Logistic Regression": baseline_results["Logistic Regression"],
    "Baseline Random Forest": baseline_results["Random Forest"],
    "Baseline Gradient Boosting": baseline_results["Gradient Boosting"],
    "Baseline SVM": baseline_results["SVM"],
    "Tuned Logistic Regression": roc_auc_score(y_val, lr_pred),
    "Tuned Random Forest": roc_auc_score(y_val, rf_pred),
    "Tuned Gradient Boosting": roc_auc_score(y_val, gb_pred),
    "Tuned SVM": roc_auc_score(y_val, svm_pred)
}

for model, score in results.items():
    print(model, "AUC:", round(score, 3))